In [27]:
import re
import polars as pl
from glob import glob

# Using a `dict` to store tables by name.

On occasion, we will need to combine more than 2 files using some combination of `UNION` and `JOIN`.  In this lecture, we will show a clean approach to scaling up these operations up to any number of files.  In the process, we will

1. Use `list` comprehensions to process and `UNION` many similar files.
2. Use `dict` comprehensions to store and access many tables by name.

## Store in `dict` or `list`?

* Natural sequence/order? $\rightarrow$ `list`
    *  Example: Lakes data and years are a natural sequence
* Easier to refer by name? $\rightarrow$ `dict`
    * Baseball files have no order and easier to refer to by name

## The Basics of working with many files.

* Use `glob.glob` to find all files that match a pattern
* Convert all files to `pd.DataFrames`
* Store the `df` in a list or dictionary

### Extracting information from a `glob` search result.

**Options.**
1. Use string methods such as `split`, or
2. Use a regular expression.

In [30]:
all_baseball_csv = glob(
    './data/baseballdatabank*/**/*.csv',
    recursive=True
)

print(all_baseball_csv)

['./data\\baseballdatabank-2023.1\\contrib\\AwardsManagers.csv', './data\\baseballdatabank-2023.1\\contrib\\AwardsPlayers.csv', './data\\baseballdatabank-2023.1\\contrib\\AwardsShareManagers.csv', './data\\baseballdatabank-2023.1\\contrib\\AwardsSharePlayers.csv', './data\\baseballdatabank-2023.1\\contrib\\CollegePlaying.csv', './data\\baseballdatabank-2023.1\\contrib\\HallOfFame.csv', './data\\baseballdatabank-2023.1\\contrib\\Salaries.csv', './data\\baseballdatabank-2023.1\\contrib\\Schools.csv', './data\\baseballdatabank-2023.1\\core\\AllstarFull.csv', './data\\baseballdatabank-2023.1\\core\\Appearances.csv', './data\\baseballdatabank-2023.1\\core\\Batting.csv', './data\\baseballdatabank-2023.1\\core\\BattingPost.csv', './data\\baseballdatabank-2023.1\\core\\Fielding.csv', './data\\baseballdatabank-2023.1\\core\\FieldingOF.csv', './data\\baseballdatabank-2023.1\\core\\FieldingOFsplit.csv', './data\\baseballdatabank-2023.1\\core\\FieldingPost.csv', './data\\baseballdatabank-2023.1\\c

#### Example 1 - Using the `str.split` method to extract a file name.

**Note.** The following cells are meant to show how the solution evolves.  In practice, this would all be proto-typed in a single cell.

In [37]:
# 1. Split on forward slash
[p.split('\\') for p in all_baseball_csv]

[['./data', 'baseballdatabank-2023.1', 'contrib', 'AwardsManagers.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'AwardsPlayers.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'AwardsShareManagers.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'AwardsSharePlayers.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'CollegePlaying.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'HallOfFame.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'Salaries.csv'],
 ['./data', 'baseballdatabank-2023.1', 'contrib', 'Schools.csv'],
 ['./data', 'baseballdatabank-2023.1', 'core', 'AllstarFull.csv'],
 ['./data', 'baseballdatabank-2023.1', 'core', 'Appearances.csv'],
 ['./data', 'baseballdatabank-2023.1', 'core', 'Batting.csv'],
 ['./data', 'baseballdatabank-2023.1', 'core', 'BattingPost.csv'],
 ['./data', 'baseballdatabank-2023.1', 'core', 'Fielding.csv'],
 ['./data', 'baseballdatabank-2023.1', 'core', 'FieldingOF.csv'],
 ['./data', 'baseballdatabank-2

In [38]:
# 1. Split on forward slash
# 2. Get the last element 
[p.split('\\')[-1] for p in all_baseball_csv]

['AwardsManagers.csv',
 'AwardsPlayers.csv',
 'AwardsShareManagers.csv',
 'AwardsSharePlayers.csv',
 'CollegePlaying.csv',
 'HallOfFame.csv',
 'Salaries.csv',
 'Schools.csv',
 'AllstarFull.csv',
 'Appearances.csv',
 'Batting.csv',
 'BattingPost.csv',
 'Fielding.csv',
 'FieldingOF.csv',
 'FieldingOFsplit.csv',
 'FieldingPost.csv',
 'HomeGames.csv',
 'Managers.csv',
 'ManagersHalf.csv',
 'Parks.csv',
 'People.csv',
 'Pitching.csv',
 'PitchingPost.csv',
 'SeriesPost.csv',
 'Teams.csv',
 'TeamsFranchises.csv',
 'TeamsHalf.csv',
 'Teams.csv']

In [39]:
# 1. Split on forward slash
# 2. Get the last element 
# 3. Split off the file type 
[p.split('\\')[-1].split('.') for p in all_baseball_csv]


[['AwardsManagers', 'csv'],
 ['AwardsPlayers', 'csv'],
 ['AwardsShareManagers', 'csv'],
 ['AwardsSharePlayers', 'csv'],
 ['CollegePlaying', 'csv'],
 ['HallOfFame', 'csv'],
 ['Salaries', 'csv'],
 ['Schools', 'csv'],
 ['AllstarFull', 'csv'],
 ['Appearances', 'csv'],
 ['Batting', 'csv'],
 ['BattingPost', 'csv'],
 ['Fielding', 'csv'],
 ['FieldingOF', 'csv'],
 ['FieldingOFsplit', 'csv'],
 ['FieldingPost', 'csv'],
 ['HomeGames', 'csv'],
 ['Managers', 'csv'],
 ['ManagersHalf', 'csv'],
 ['Parks', 'csv'],
 ['People', 'csv'],
 ['Pitching', 'csv'],
 ['PitchingPost', 'csv'],
 ['SeriesPost', 'csv'],
 ['Teams', 'csv'],
 ['TeamsFranchises', 'csv'],
 ['TeamsHalf', 'csv'],
 ['Teams', 'csv']]

In [40]:
# 1. Split on forward slash
# 2. Get the last element 
# 3. Split off the file type 
# 4. Get the first enter (e.g. file name)
[p.split('\\')[-1].split('.')[0] for p in all_baseball_csv]

['AwardsManagers',
 'AwardsPlayers',
 'AwardsShareManagers',
 'AwardsSharePlayers',
 'CollegePlaying',
 'HallOfFame',
 'Salaries',
 'Schools',
 'AllstarFull',
 'Appearances',
 'Batting',
 'BattingPost',
 'Fielding',
 'FieldingOF',
 'FieldingOFsplit',
 'FieldingPost',
 'HomeGames',
 'Managers',
 'ManagersHalf',
 'Parks',
 'People',
 'Pitching',
 'PitchingPost',
 'SeriesPost',
 'Teams',
 'TeamsFranchises',
 'TeamsHalf',
 'Teams']

In [42]:
# 1. Split on forward slash
# 2. Get the last element 
# 3. Split off the file type 
# 4. Get the first enter (e.g. file name)
# 5. Refactor
get_file_name = lambda p: p.split('\\')[-1].split('.')[0]

[get_file_name(p) for p in all_baseball_csv]


['AwardsManagers',
 'AwardsPlayers',
 'AwardsShareManagers',
 'AwardsSharePlayers',
 'CollegePlaying',
 'HallOfFame',
 'Salaries',
 'Schools',
 'AllstarFull',
 'Appearances',
 'Batting',
 'BattingPost',
 'Fielding',
 'FieldingOF',
 'FieldingOFsplit',
 'FieldingPost',
 'HomeGames',
 'Managers',
 'ManagersHalf',
 'Parks',
 'People',
 'Pitching',
 'PitchingPost',
 'SeriesPost',
 'Teams',
 'TeamsFranchises',
 'TeamsHalf',
 'Teams']

#### Example 2 - Using a regular expression to capture the file name.

In [44]:
# 1. Start with an example path
file_name = re.compile(r'./data/baseballdatabank-2023.1/core/Managers.csv')
                       
[_match for p in all_baseball_csv if (_match := file_name.match(p))]

[]

In [45]:
# 1. Start with an example path
# 2. Make it match any file name

file_name = re.compile(r'./data/baseballdatabank-2023.1/core/[a-zA-Z]+.csv')

[_match for p in all_baseball_csv if (_match := file_name.match(p))]

[]

In [13]:
# 1. Start with an example path
# 2. Make it match any file name
# 3. Make it match any subfolder

file_name = re.compile(r'./data/baseballdatabank-2023.1/[a-z]+/[a-zA-Z]+.csv')

[_match for p in all_baseball_csv if (_match := file_name.match(p))]

[]

In [14]:
# 1. Start with an example path
# 2. Make it match any file name
# 3. Make it match any subfolder
# 4. Add a capture group and extract the data

file_name = re.compile(r'./data/baseballdatabank-2023.1/[a-z]+/([a-zA-Z]+).csv')

[_match.group(1) for p in all_baseball_csv if (_match := file_name.match(p))]

[]

In [15]:
# 1. Start with an example path
# 2. Make it match any file name
# 3. Make it match any subfolder
# 4. Add a capture group and extract the data
# 5. Refactor

file_name = re.compile(r'./data/baseballdatabank-2023.1/[a-z]+/([a-zA-Z]+).csv')

get_file_names = lambda paths: [_match.group(1) for p in paths if (_match := file_name.match(p))]

get_file_names(all_baseball_csv)

[]

### Dealing with Windows paths

* Windows uses backslash `\` instead of forward slash `/` to separate folders/files.
* Even on Windows, `glob` understands unix style paths that use `/` to separate files/folders.
* Since `\` is the escape character in Python, Windows paths will contain an escaped/literal backslash `\\`.
* The wild card parts of the pattern will return Windows style paths with `\\`

In [48]:
# Non-recursive search run on Windows
baseball_core_csv = glob('./data/baseballdatabank*/core/*csv')#, recursive=True)

baseball_core_csv

['./data\\baseballdatabank-2023.1\\core\\AllstarFull.csv',
 './data\\baseballdatabank-2023.1\\core\\Appearances.csv',
 './data\\baseballdatabank-2023.1\\core\\Batting.csv',
 './data\\baseballdatabank-2023.1\\core\\BattingPost.csv',
 './data\\baseballdatabank-2023.1\\core\\Fielding.csv',
 './data\\baseballdatabank-2023.1\\core\\FieldingOF.csv',
 './data\\baseballdatabank-2023.1\\core\\FieldingOFsplit.csv',
 './data\\baseballdatabank-2023.1\\core\\FieldingPost.csv',
 './data\\baseballdatabank-2023.1\\core\\HomeGames.csv',
 './data\\baseballdatabank-2023.1\\core\\Managers.csv',
 './data\\baseballdatabank-2023.1\\core\\ManagersHalf.csv',
 './data\\baseballdatabank-2023.1\\core\\Parks.csv',
 './data\\baseballdatabank-2023.1\\core\\People.csv',
 './data\\baseballdatabank-2023.1\\core\\Pitching.csv',
 './data\\baseballdatabank-2023.1\\core\\PitchingPost.csv',
 './data\\baseballdatabank-2023.1\\core\\SeriesPost.csv',
 './data\\baseballdatabank-2023.1\\core\\Teams.csv',
 './data\\baseballdataba

In [47]:
# Recursive search run on Windows
all_baseball_csv = glob('./data/baseballdatabank*/**/*.csv', recursive=True)

all_baseball_csv

['./data\\baseballdatabank-2023.1\\contrib\\AwardsManagers.csv',
 './data\\baseballdatabank-2023.1\\contrib\\AwardsPlayers.csv',
 './data\\baseballdatabank-2023.1\\contrib\\AwardsShareManagers.csv',
 './data\\baseballdatabank-2023.1\\contrib\\AwardsSharePlayers.csv',
 './data\\baseballdatabank-2023.1\\contrib\\CollegePlaying.csv',
 './data\\baseballdatabank-2023.1\\contrib\\HallOfFame.csv',
 './data\\baseballdatabank-2023.1\\contrib\\Salaries.csv',
 './data\\baseballdatabank-2023.1\\contrib\\Schools.csv',
 './data\\baseballdatabank-2023.1\\core\\AllstarFull.csv',
 './data\\baseballdatabank-2023.1\\core\\Appearances.csv',
 './data\\baseballdatabank-2023.1\\core\\Batting.csv',
 './data\\baseballdatabank-2023.1\\core\\BattingPost.csv',
 './data\\baseballdatabank-2023.1\\core\\Fielding.csv',
 './data\\baseballdatabank-2023.1\\core\\FieldingOF.csv',
 './data\\baseballdatabank-2023.1\\core\\FieldingOFsplit.csv',
 './data\\baseballdatabank-2023.1\\core\\FieldingPost.csv',
 './data\\baseballda

### Extracting information from Windows paths

On Windows, we need to switch `/` to `\\`

#### Example 1 - Split and get the file name in Windows

In [49]:
get_file_name = lambda p: p.split('\\')[-1].split('.')[0]

[get_file_name(p) for p in all_baseball_csv]


['AwardsManagers',
 'AwardsPlayers',
 'AwardsShareManagers',
 'AwardsSharePlayers',
 'CollegePlaying',
 'HallOfFame',
 'Salaries',
 'Schools',
 'AllstarFull',
 'Appearances',
 'Batting',
 'BattingPost',
 'Fielding',
 'FieldingOF',
 'FieldingOFsplit',
 'FieldingPost',
 'HomeGames',
 'Managers',
 'ManagersHalf',
 'Parks',
 'People',
 'Pitching',
 'PitchingPost',
 'SeriesPost',
 'Teams',
 'TeamsFranchises',
 'TeamsHalf',
 'Teams']

#### Example 2 - Using a regular expression to extract file name in Windows

In [50]:
file_name = re.compile(r'./data\\baseballdatabank-2023.1\\[a-z]+\\([a-zA-Z]+).csv')

get_file_names = lambda paths: [_match.group(1) for p in paths if (_match := file_name.match(p))]

get_file_names(all_baseball_csv)

['AwardsManagers',
 'AwardsPlayers',
 'AwardsShareManagers',
 'AwardsSharePlayers',
 'CollegePlaying',
 'HallOfFame',
 'Salaries',
 'Schools',
 'AllstarFull',
 'Appearances',
 'Batting',
 'BattingPost',
 'Fielding',
 'FieldingOF',
 'FieldingOFsplit',
 'FieldingPost',
 'HomeGames',
 'Managers',
 'ManagersHalf',
 'Parks',
 'People',
 'Pitching',
 'PitchingPost',
 'SeriesPost',
 'Teams',
 'TeamsFranchises',
 'TeamsHalf',
 'Teams']

## Storing tables by name in a `dict`

### Example 1 - Read all baseball database using `dict`

**Task:** Create a `dict` of tables for all tables in the Lahman database

#### Step 1 - Use `glob` to find paths for all CSV files

In [60]:
from glob import glob

files = glob('./data/baseballdatabank*/core/*.csv')
print(files)

['./data\\baseballdatabank-2023.1\\core\\AllstarFull.csv', './data\\baseballdatabank-2023.1\\core\\Appearances.csv', './data\\baseballdatabank-2023.1\\core\\Batting.csv', './data\\baseballdatabank-2023.1\\core\\BattingPost.csv', './data\\baseballdatabank-2023.1\\core\\Fielding.csv', './data\\baseballdatabank-2023.1\\core\\FieldingOF.csv', './data\\baseballdatabank-2023.1\\core\\FieldingOFsplit.csv', './data\\baseballdatabank-2023.1\\core\\FieldingPost.csv', './data\\baseballdatabank-2023.1\\core\\HomeGames.csv', './data\\baseballdatabank-2023.1\\core\\Managers.csv', './data\\baseballdatabank-2023.1\\core\\ManagersHalf.csv', './data\\baseballdatabank-2023.1\\core\\Parks.csv', './data\\baseballdatabank-2023.1\\core\\People.csv', './data\\baseballdatabank-2023.1\\core\\Pitching.csv', './data\\baseballdatabank-2023.1\\core\\PitchingPost.csv', './data\\baseballdatabank-2023.1\\core\\SeriesPost.csv', './data\\baseballdatabank-2023.1\\core\\Teams.csv', './data\\baseballdatabank-2023.1\\core\\

#### Step 2 - Make a function to extract the table name

In [61]:
import re
files = [p.replace('\\', '/') for p in files]

FILE_NAME_RE = re.compile(r'^\./data/baseballdatabank.*/core/([A-Za-z0-9_]+)\.csv$')
file_name = lambda p: FILE_NAME_RE.match(p).group(1) if FILE_NAME_RE.match(p) else None

print([file_name(p) for p in files])

['AllstarFull', 'Appearances', 'Batting', 'BattingPost', 'Fielding', 'FieldingOF', 'FieldingOFsplit', 'FieldingPost', 'HomeGames', 'Managers', 'ManagersHalf', 'Parks', 'People', 'Pitching', 'PitchingPost', 'SeriesPost', 'Teams', 'TeamsFranchises', 'TeamsHalf']


#### 4 - Read in the tables.

In [62]:
(baseball_db := 
 {file_name(p):pl.read_csv(p, infer_schema_length=10000) for p in files}
)

{'AllstarFull': shape: (5_516, 8)
 ┌───────────┬────────┬─────────┬──────────────┬────────┬──────┬─────┬─────────────┐
 │ playerID  ┆ yearID ┆ gameNum ┆ gameID       ┆ teamID ┆ lgID ┆ GP  ┆ startingPos │
 │ ---       ┆ ---    ┆ ---     ┆ ---          ┆ ---    ┆ ---  ┆ --- ┆ ---         │
 │ str       ┆ i64    ┆ i64     ┆ str          ┆ str    ┆ str  ┆ i64 ┆ i64         │
 ╞═══════════╪════════╪═════════╪══════════════╪════════╪══════╪═════╪═════════════╡
 │ gomezle01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ NYA    ┆ AL   ┆ 1   ┆ 1           │
 │ ferreri01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ BOS    ┆ AL   ┆ 1   ┆ 2           │
 │ gehrilo01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ NYA    ┆ AL   ┆ 1   ┆ 3           │
 │ gehrich01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ DET    ┆ AL   ┆ 1   ┆ 4           │
 │ dykesji01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ CHA    ┆ AL   ┆ 1   ┆ 5           │
 │ …         ┆ …      ┆ …       ┆ …            ┆ …      ┆ …    ┆ …   ┆ …           │
 │ rileyau01 ┆ 2022   ┆ 0      

### We can now access all the tables by name.

In [63]:
# Biggish output
baseball_db

{'AllstarFull': shape: (5_516, 8)
 ┌───────────┬────────┬─────────┬──────────────┬────────┬──────┬─────┬─────────────┐
 │ playerID  ┆ yearID ┆ gameNum ┆ gameID       ┆ teamID ┆ lgID ┆ GP  ┆ startingPos │
 │ ---       ┆ ---    ┆ ---     ┆ ---          ┆ ---    ┆ ---  ┆ --- ┆ ---         │
 │ str       ┆ i64    ┆ i64     ┆ str          ┆ str    ┆ str  ┆ i64 ┆ i64         │
 ╞═══════════╪════════╪═════════╪══════════════╪════════╪══════╪═════╪═════════════╡
 │ gomezle01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ NYA    ┆ AL   ┆ 1   ┆ 1           │
 │ ferreri01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ BOS    ┆ AL   ┆ 1   ┆ 2           │
 │ gehrilo01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ NYA    ┆ AL   ┆ 1   ┆ 3           │
 │ gehrich01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ DET    ┆ AL   ┆ 1   ┆ 4           │
 │ dykesji01 ┆ 1933   ┆ 0       ┆ ALS193307060 ┆ CHA    ┆ AL   ┆ 1   ┆ 5           │
 │ …         ┆ …      ┆ …       ┆ …            ┆ …      ┆ …    ┆ …   ┆ …           │
 │ rileyau01 ┆ 2022   ┆ 0      

In [64]:
baseball_db['Teams']

yearID,lgID,teamID,franchID,divID,Rank,G,Ghome,W,L,DivWin,WCWin,LgWin,WSWin,R,AB,H,2B,3B,HR,BB,SO,SB,CS,HBP,SF,RA,ER,ERA,CG,SHO,SV,IPouts,HA,HRA,BBA,SOA,E,DP,FP,name,park,attendance,BPF,PPF,teamIDBR,teamIDlahman45,teamIDretro
i64,str,str,str,str,i64,i64,i64,i64,i64,str,str,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,f64,str,str,i64,i64,i64,str,str,str
1871,"""NA""","""BS1""","""BNA""",null,3,31,null,20,10,null,null,"""N""",null,401,1372,426,70,37,3,60,19,73,16,null,null,303,109,3.55,22,1,3,828,367,2,42,23,243,24,0.834,"""Boston Red Stockings""","""South End Grounds I""",null,103,98,"""BOS""","""BS1""","""BS1"""
1871,"""NA""","""CH1""","""CNA""",null,2,28,null,19,9,null,null,"""N""",null,302,1196,323,52,21,10,60,22,69,21,null,null,241,77,2.76,25,0,1,753,308,6,28,22,229,16,0.829,"""Chicago White Stockings""","""Union Base-Ball Grounds""",null,104,102,"""CHI""","""CH1""","""CH1"""
1871,"""NA""","""CL1""","""CFC""",null,8,29,null,10,19,null,null,"""N""",null,249,1186,328,35,40,7,26,25,18,8,null,null,341,116,4.11,23,0,0,762,346,13,53,34,234,15,0.818,"""Cleveland Forest Citys""","""National Association Grounds""",null,96,100,"""CLE""","""CL1""","""CL1"""
1871,"""NA""","""FW1""","""KEK""",null,7,19,null,7,12,null,null,"""N""",null,137,746,178,19,8,2,33,9,16,4,null,null,243,97,5.17,19,1,0,507,261,5,21,17,163,8,0.803,"""Fort Wayne Kekiongas""","""Hamilton Field""",null,101,107,"""KEK""","""FW1""","""FW1"""
1871,"""NA""","""NY2""","""NNA""",null,5,33,null,16,17,null,null,"""N""",null,302,1404,403,43,21,1,33,15,46,15,null,null,313,121,3.72,32,1,0,879,373,7,42,22,235,14,0.84,"""New York Mutuals""","""Union Grounds (Brooklyn)""",null,90,88,"""NYU""","""NY2""","""NY2"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2022,"""NL""","""SLN""","""STL""","""C""",1,162,81,93,69,"""Y""","""N""","""N""","""N""",772,5496,1386,290,21,197,537,1226,95,25,80,45,637,605,3.79,3,17,37,4307,1335,146,489,1177,66,181,0.989,"""St. Louis Cardinals""","""Busch Stadium III""",3320551,94,94,"""STL""","""SLN""","""SLN"""
2022,"""AL""","""TBA""","""TBD""","""E""",1,162,81,86,76,"""N""","""Y""","""N""","""N""",666,5412,1294,296,17,139,500,1395,95,37,57,31,614,544,3.41,0,10,44,4307,1260,172,384,1384,84,110,0.985,"""Tampa Bay Rays""","""Tropicana Field""",1128127,95,93,"""TBR""","""TBA""","""TBA"""
2022,"""AL""","""TEX""","""TEX""","""W""",4,162,81,68,94,"""N""","""N""","""N""","""N""",707,5478,1308,224,20,198,456,1446,128,41,47,38,743,673,4.22,1,10,37,4305,1345,169,581,1314,96,143,0.984,"""Texas Rangers""","""Globe Life Field""",2011361,100,101,"""TEX""","""TEX""","""TEX"""


In [65]:
baseball_db['Batting']

playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP
str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,i64,i64,str,i64
"""abercda01""",1871,1,"""TRO""","""NA""",1,4,0,0,0,0,0,0,0,0,0,0,null,null,null,null,0
"""addybo01""",1871,1,"""RC1""","""NA""",25,118,30,32,6,0,0,13,8,1,4,0,null,null,null,null,0
"""allisar01""",1871,1,"""CL1""","""NA""",29,137,28,40,4,5,0,19,3,1,2,5,null,null,null,null,1
"""allisdo01""",1871,1,"""WS3""","""NA""",27,133,28,44,10,2,2,27,1,1,0,2,null,null,null,null,0
"""ansonca01""",1871,1,"""RC1""","""NA""",25,120,29,39,11,3,0,16,6,2,2,1,null,null,null,null,0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""zimmebr01""",2022,1,"""TOR""","""AL""",77,76,11,8,4,0,2,3,2,1,5,33,"""0""",5,1,"""0""",0
"""zimmebr01""",2022,2,"""PHI""","""NL""",9,16,4,4,1,0,0,0,0,0,0,4,"""0""",0,0,"""0""",0
"""zimmebr01""",2022,3,"""TOR""","""AL""",23,13,3,1,0,0,0,2,1,1,0,8,"""0""",1,0,"""0""",0


## Example 2 - Reading and joining the baseball database using `dict`

**Task:** Collect the number of total hits for each batters in the 2010 season join on their first and last name.

In the second example, we will store the data frames in a `dict`, which will make it easier to join the files by ne

#### Step 1 - Get the files names

* Only need the `Batting.csv` and `People.csv`.  
* Narrow with a RegEx

In [67]:

from glob import glob

files = glob('./data/baseballdatabank*/core/*.csv')
print(files)

['./data\\baseballdatabank-2023.1\\core\\AllstarFull.csv', './data\\baseballdatabank-2023.1\\core\\Appearances.csv', './data\\baseballdatabank-2023.1\\core\\Batting.csv', './data\\baseballdatabank-2023.1\\core\\BattingPost.csv', './data\\baseballdatabank-2023.1\\core\\Fielding.csv', './data\\baseballdatabank-2023.1\\core\\FieldingOF.csv', './data\\baseballdatabank-2023.1\\core\\FieldingOFsplit.csv', './data\\baseballdatabank-2023.1\\core\\FieldingPost.csv', './data\\baseballdatabank-2023.1\\core\\HomeGames.csv', './data\\baseballdatabank-2023.1\\core\\Managers.csv', './data\\baseballdatabank-2023.1\\core\\ManagersHalf.csv', './data\\baseballdatabank-2023.1\\core\\Parks.csv', './data\\baseballdatabank-2023.1\\core\\People.csv', './data\\baseballdatabank-2023.1\\core\\Pitching.csv', './data\\baseballdatabank-2023.1\\core\\PitchingPost.csv', './data\\baseballdatabank-2023.1\\core\\SeriesPost.csv', './data\\baseballdatabank-2023.1\\core\\Teams.csv', './data\\baseballdatabank-2023.1\\core\\

#### Step 2 - Make helper functions to get the name from path

In [75]:
import re

# normalize paths to forward slashes
files = [p.replace('\\', '/') for p in files]

FILE_NAME_RE = re.compile(r'^\./data/baseballdatabank.*/core/([A-Za-z0-9_]+)\.csv$')

# helper: extract the name, or None if no match
file_name = lambda p: (m.group(1) if (m := FILE_NAME_RE.match(p)) else None)

# filter only Batting or People
is_batting_or_people = lambda p: file_name(p) in {"Batting", "People"}

result = [file_name(p) for p in files if is_batting_or_people(p)]
print(result)


['Batting', 'People']


#### Step 3 - Use a comprehension to read in all files

**Note:** The data is small (< 10mb total) so it is safe to read all at once.

In [76]:
dfs = {file_name(p):pl.read_csv(p) for p in files if is_batting_or_people(p)}
dfs['Batting'].head()

playerID,yearID,stint,teamID,lgID,G,AB,R,H,2B,3B,HR,RBI,SB,CS,BB,SO,IBB,HBP,SH,SF,GIDP
str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,str,str,str,i64
"""abercda01""",1871,1,"""TRO""","""NA""",1,4,0,0,0,0,0,0,0,0,0,0,null,null,null,null,0
"""addybo01""",1871,1,"""RC1""","""NA""",25,118,30,32,6,0,0,13,8,1,4,0,null,null,null,null,0
"""allisar01""",1871,1,"""CL1""","""NA""",29,137,28,40,4,5,0,19,3,1,2,5,null,null,null,null,1
"""allisdo01""",1871,1,"""WS3""","""NA""",27,133,28,44,10,2,2,27,1,1,0,2,null,null,null,null,0
"""ansonca01""",1871,1,"""RC1""","""NA""",25,120,29,39,11,3,0,16,6,2,2,1,null,null,null,null,0


In [79]:
dfs['People'].head()

playerID,birthYear,birthMonth,birthDay,birthCountry,birthState,birthCity,deathYear,deathMonth,deathDay,deathCountry,deathState,deathCity,nameFirst,nameLast,nameGiven,weight,height,bats,throws,debut,finalGame,retroID,bbrefID
str,i64,i64,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str
"""aardsda01""",1981,12,27,"""USA""","""CO""","""Denver""",null,null,null,null,null,null,"""David""","""Aardsma""","""David Allan""",215,75,"""R""","""R""","""2004-04-06""","""2015-08-23""","""aardd001""","""aardsda01"""
"""aaronha01""",1934,2,5,"""USA""","""AL""","""Mobile""",2021,1,22,"""USA""","""GA""","""Atlanta""","""Hank""","""Aaron""","""Henry Louis""",180,72,"""R""","""R""","""1954-04-13""","""1976-10-03""","""aaroh101""","""aaronha01"""
"""aaronto01""",1939,8,5,"""USA""","""AL""","""Mobile""",1984,8,16,"""USA""","""GA""","""Atlanta""","""Tommie""","""Aaron""","""Tommie Lee""",190,75,"""R""","""R""","""1962-04-10""","""1971-09-26""","""aarot101""","""aaronto01"""
"""aasedo01""",1954,9,8,"""USA""","""CA""","""Orange""",null,null,null,null,null,null,"""Don""","""Aase""","""Donald William""",190,75,"""R""","""R""","""1977-07-26""","""1990-10-03""","""aased001""","""aasedo01"""
"""abadan01""",1972,8,25,"""USA""","""FL""","""Palm Beach""",null,null,null,null,null,null,"""Andy""","""Abad""","""Fausto Andres""",184,73,"""L""","""L""","""2001-09-10""","""2006-04-13""","""abada001""","""abadan01"""


#### Step 4 - Preprocess each file.

In [80]:
# Filter, select, and aggregate hits for 2010.
(hits_in_2010_raw := 
 dfs['Batting']
.select(['yearID', 'playerID', 'H'])
.filter(pl.col('yearID') == 2010)
.group_by('playerID')
.agg(pl.col('H').mean().alias('Total Hits'))
).head()

playerID,Total Hits
str,f64
"""smithjo06""",0.0
"""markani01""",187.0
"""cabreev01""",44.0
"""weaveje02""",2.0
"""fisheca01""",0.0


In [81]:
# Grab the first and last names from People.

(player_names := 
 dfs['People']
 .select(['playerID', 'nameFirst', 'nameLast'])
).head(2)

playerID,nameFirst,nameLast
str,str,str
"""aardsda01""","""David""","""Aardsma"""
"""aaronha01""","""Hank""","""Aaron"""


#### Step 4 -- Join the tables

In [82]:
(hits_in_2010 := 
 hits_in_2010_raw 
 .join(player_names, on='playerID', how='left')
 .drop('playerID')
).head()

Total Hits,nameFirst,nameLast
f64,str,str
0.0,"""Jordan""","""Smith"""
187.0,"""Nick""","""Markakis"""
44.0,"""Everth""","""Cabrera"""
2.0,"""Jered""","""Weaver"""
0.0,"""Carlos""","""Fisher"""


## <font color="red"> Exercise 3.2 </font>

We want to get the total hits allowed for all pitchers during the 2000-2010 seasons.  Use `glob` and a `dict` to collect this information into a table that includes the players first and last names.

In [101]:
# Your code here
import re

# normalize paths to forward slashes
files = [p.replace('\\', '/') for p in files]

FILE_NAME_RE = re.compile(r'^\./data/baseballdatabank.*/core/([A-Za-z0-9_]+)\.csv$')

# helper: extract the name, or None if no match
file_name = lambda p: (m.group(1) if (m := FILE_NAME_RE.match(p)) else None)

# filter only Batting or People
is_pitching_or_people = lambda p: file_name(p) in {"Pitching", "People"}

result = [file_name(p) for p in files if is_pitching_or_people(p)]
print(result)

['People', 'Pitching']


In [102]:
dfs = {file_name(p):pl.read_csv(p) for p in files if is_pitching_or_people(p)}
dfs['Pitching'].head()

playerID,yearID,stint,teamID,lgID,W,L,G,GS,CG,SHO,SV,IPouts,H,ER,HR,BB,SO,BAOpp,ERA,IBB,WP,HBP,BK,BFP,GF,R,SH,SF,GIDP
str,i64,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,str,f64,str,i64,str,i64,i64,i64,i64,str,str,str
"""bechtge01""",1871,1,"""PH1""","""NA""",1,2,3,3,2,0,0,78,43,23,0,11,1,null,7.96,null,7,null,0,146,0,42,null,null,null
"""brainas01""",1871,1,"""WS3""","""NA""",12,15,30,30,30,0,0,792,361,132,4,37,13,null,4.5,null,7,null,0,1291,0,292,null,null,null
"""fergubo01""",1871,1,"""NY2""","""NA""",0,0,1,0,0,0,0,3,8,3,0,0,0,null,27.0,null,2,null,0,14,0,9,null,null,null
"""fishech01""",1871,1,"""RC1""","""NA""",4,16,24,24,22,1,0,639,295,103,3,31,15,null,4.35,null,20,null,0,1080,1,257,null,null,null
"""fleetfr01""",1871,1,"""NY2""","""NA""",0,1,1,1,1,0,0,27,20,10,0,3,0,null,10.0,null,0,null,0,57,0,21,null,null,null


In [92]:
dfs['People'].head()

playerID,birthYear,birthMonth,birthDay,birthCountry,birthState,birthCity,deathYear,deathMonth,deathDay,deathCountry,deathState,deathCity,nameFirst,nameLast,nameGiven,weight,height,bats,throws,debut,finalGame,retroID,bbrefID
str,i64,i64,i64,str,str,str,i64,i64,i64,str,str,str,str,str,str,i64,i64,str,str,str,str,str,str
"""aardsda01""",1981,12,27,"""USA""","""CO""","""Denver""",null,null,null,null,null,null,"""David""","""Aardsma""","""David Allan""",215,75,"""R""","""R""","""2004-04-06""","""2015-08-23""","""aardd001""","""aardsda01"""
"""aaronha01""",1934,2,5,"""USA""","""AL""","""Mobile""",2021,1,22,"""USA""","""GA""","""Atlanta""","""Hank""","""Aaron""","""Henry Louis""",180,72,"""R""","""R""","""1954-04-13""","""1976-10-03""","""aaroh101""","""aaronha01"""
"""aaronto01""",1939,8,5,"""USA""","""AL""","""Mobile""",1984,8,16,"""USA""","""GA""","""Atlanta""","""Tommie""","""Aaron""","""Tommie Lee""",190,75,"""R""","""R""","""1962-04-10""","""1971-09-26""","""aarot101""","""aaronto01"""
"""aasedo01""",1954,9,8,"""USA""","""CA""","""Orange""",null,null,null,null,null,null,"""Don""","""Aase""","""Donald William""",190,75,"""R""","""R""","""1977-07-26""","""1990-10-03""","""aased001""","""aasedo01"""
"""abadan01""",1972,8,25,"""USA""","""FL""","""Palm Beach""",null,null,null,null,null,null,"""Andy""","""Abad""","""Fausto Andres""",184,73,"""L""","""L""","""2001-09-10""","""2006-04-13""","""abada001""","""abadan01"""


In [103]:
(player_namess := 
 dfs['People']
 .select(['playerID', 'nameFirst', 'nameLast'])
).head(2)

playerID,nameFirst,nameLast
str,str,str
"""aardsda01""","""David""","""Aardsma"""
"""aaronha01""","""Hank""","""Aaron"""


In [104]:
(hits_in_2010s := 
 dfs['Pitching']
.select(['yearID', 'playerID', 'H'])
.filter((pl.col('yearID') >= 2010) & (pl.col('yearID') <= 2020))
.group_by('playerID')
.agg(pl.col('H').mean().alias('Total Hits'))
).head()

playerID,Total Hits
str,f64
"""jimened01""",12.0
"""sanchan01""",136.166667
"""archech01""",125.222222
"""slanida01""",0.0
"""cedenxa01""",14.583333


In [105]:
(hits_in_teens := 
 hits_in_2010s
 .join(player_namess, on='playerID', how='left')
 .drop('playerID')
 .sort("Total Hits", descending=True)
).head()

Total Hits,nameFirst,nameLast
f64,str,str
221.5,"""Mark""","""Buehrle"""
192.6,"""Hiroki""","""Kuroda"""
190.363636,"""Rick""","""Porcello"""
189.666667,"""Carl""","""Pavano"""
189.25,"""R. A.""","""Dickey"""
